# 🟡 미션 2 — 반복 붕괴를 일부러 만들고, 풀어 본다

수업 `## 3` 에서 본 그 현상을 **내 손으로 만들었다가 고친다.**

## 낼 것

1. **붕괴한 결과** 캡처 1장
2. **풀린 결과** 캡처 1장 + 그때 쓴 `repetition_penalty` 값

In [1]:
import torch
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel

MODEL_ID = "skt/kogpt2-base-v2"

tokenizer = PreTrainedTokenizerFast.from_pretrained(
    MODEL_ID,
    bos_token="</s>", eos_token="</s>",
    unk_token="<unk>", pad_token="<pad>", mask_token="<mask>",
)
model = GPT2LMHeadModel.from_pretrained(MODEL_ID).eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)


def generate(prompt, max_new_tokens=80, **options):
    ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new_tokens,
                             pad_token_id=tokenizer.pad_token_id, **options)
    return tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):].strip()


print("준비 완료 :", device)

/home/student/llm-practice/c2-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 149/149 [00:00<00:00, 16676.04it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


준비 완료 : cuda


## 1. 붕괴하는 프롬프트를 찾는다

**greedy 로 길게 뽑으면** 대부분 어딘가에서 같은 말을 맴돌기 시작한다.
아래 후보를 돌려 보고, 가장 심하게 맴도는 것을 고른다. 내 문장을 넣어도 된다.

In [2]:
후보들 = [
    "깊은 산속에 작은 집이",
    "커피를 마시면",
    "한국의 수도는",
    "인공지능 기술이 발전하면서",
]

for p in 후보들:
    print("=" * 60)
    print(f"[{p}]")
    print(generate(p, do_sample=False))
    print()

[깊은 산속에 작은 집이]
하나 있다.
바로 이 집은 바로 이 산속에 있는 작은 집이다.
이 집은 바로 이 산속에 있는 작은 집이다.
이 집은 바로 이 산속에 있는 작은 집이다.
이 집은 바로 이 산속에 있는 작은 집이다.
이 집은 바로 이 산속에 있는 작은 집이다.
이 집은 바로 이 산속에 있는 작은 집이다.
이 집은 바로 이 산속에 있는 작은 집이다.
이 집은 바로 이 산속에 있는

[커피를 마시면]
기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서 기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서 기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서 기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서 기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서 기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서 기분이

[한국의 수도는]
서울이다.
서울은 서울이 아니라 서울이다.
서울은 서울이 아니라 서울이다.
서울은 서울이 아니라 서울이다.
서울은 서울이 아니다.
서울은 서울이 아니다.
서울은 서울이 아니다.
서울은 서울이 아니다.
서울은 서울이 아니다.
서울은 서울이 아니다.
서울은 서울이 아니다.
서울은 서울이 아니다.
서울은 서울이 아니다.
서울은 서울이 아니다.
서울은 서울이 아니다.
서울은

[인공지능 기술이 발전하면서]
인간의 뇌는 더 이상 인간의 뇌가 아니다.
인간의 뇌는 인간의 뇌보다 더 복잡한 뇌 구조를 갖고 있다.
뇌는 인간의 뇌보다 더 복잡한 뇌 구조를 갖고 있다.
뇌는 인간의 뇌보다 더 복잡한 뇌 구조를 갖고 있다.
뇌는 인간의 뇌보다 더 복잡한 뇌 구조를 갖고 있다.
뇌는 인간의 뇌보다 더 복잡한 뇌 구조를 갖고 있다.
뇌는 인간의 뇌보다 더 복잡한 뇌 구조를 갖고 있다.
뇌는 인간의 뇌



## 2. 제일 심한 것을 골라 붕괴를 확인한다

In [3]:
내_프롬프트 = "커피를 마시면"          # ← 위에서 제일 심했던 것으로 바꾼다

붕괴 = generate(내_프롬프트, max_new_tokens=100, do_sample=False)
print("--- 붕괴 (repetition_penalty 없음) ---")
print(붕괴)

--- 붕괴 (repetition_penalty 없음) ---
기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서 기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서 기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서 기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서 기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서 기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서 기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서 기분이 좋아지는 것 같습니다.
이렇게


**📸 여기를 캡처한다** (제출물 1)

## 3. 벌칙을 조금씩 올려 본다

`repetition_penalty` 는 **이미 나온 토큰의 점수를 깎는** 값이다.
1.0이면 벌칙 없음. 올릴수록 세게 깎는다.

In [4]:
for penalty in [1.0, 1.05, 1.1, 1.2, 1.5, 2.0]:
    print("=" * 60)
    print(f"[repetition_penalty={penalty}]")
    print(generate(내_프롬프트, max_new_tokens=100,
                   do_sample=False, repetition_penalty=penalty))
    print()

[repetition_penalty=1.0]
기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서 기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서 기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서 기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서 기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서 기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서 기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서 기분이 좋아지는 것 같습니다.
이렇게

[repetition_penalty=1.05]
기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서, 또 다른 즐거움을 느낄 수 있는 시간이 될 것 같습니다.
오늘은 어떤 음식들을 먹었는지 살펴보겠습니다.
먼저 오늘은 뭘 먹었는지 살펴보겠습니다.
우선 오늘은 뭘 먹었는지 살펴보겠습니다.
먼저 오늘의 메뉴는 뭘 먹었는지 살펴보겠습니다.
먼저 오늘의 대표메뉴는 뭘 먹었는지 살펴보겠습니다.
먼저 오늘의 대표메뉴는 뭘 먹었

[repetition_penalty=1.1]
기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서, 또 다른 즐거움을 느낄 수 있는 시간이 될 것 같습니다.
오늘은 어떤 음식들을 즐겨보셨나요?
먼저 오늘은요.
바로 햄버거입니다.
햄버거는 정말 맛있어요.
그런데 이게 진짜 맛있다고 하더라고요.
그래서 저는 햄버거를 먹고 나서 바로 햄버거를 먹었거든요.
그리고 그 다음에 햄버거를 먹으면서, 그리고 또 다른 즐거움

[repetition_penalty=1.2]
기분이 좋아지는 것 같습니다.
이렇게 맛있는 음식을 먹으면서, 또 다른 즐거움을 느낄 수 있는 시간이 될 거라고 생각합니다.
오늘은 어떤 음식들을 먹어보셨나요?
저는 오늘도 햄버거와 치킨을 먹고 싶었어요.
그런데 제가 좋아하는 메뉴는 바로 닭가슴살이었거든요.
닭가슴살은 정말 맛있어서 너무 좋았죠.
그리고 저는 이번에 먹은 닭가슴살을 먹기 전에 먼저 닭가슴살에 대한 이야기를 해봤는데

## 4. 읽고 고른다 — 찾을 것은 **루프가 풀리는 지점**이다

값을 순서대로 읽으면서 **어디서 루프가 풀렸는지** 표시한다.

그리고 한 가지 더 확인한다 —

> **풀린 지점 위로는 결과가 얼마나 달라지는가?**

값을 두 배로 올리면 두 배로 달라질 것 같지만, 실제로는 **어느 지점부터 거의 안 바뀔 수 있다.**
한번 반복을 피해 다른 길로 들어서고 나면, 더 세게 깎아도 갈 길이 이미 갈렸기 때문이다.

직접 확인해 보고 아래에 적는다. (내 프롬프트·길이에 따라 다르게 나올 수 있다)

**📸 루프가 풀린 값의 결과를 캡처한다** (제출물 2)

## 제출

```
내 프롬프트 :
붕괴했을 때 맴돌던 구절 :
루프가 풀린 repetition_penalty 값 :
그 위로 더 올렸을 때 달라졌나 (예/아니오 + 한 줄) :
```

## 🔵 더 해 보고 싶다면 — 정해진 답 없음

- **반복 붕괴를 가장 빨리 일으키는 프롬프트**를 찾아 본다.
  짧은 게 유리할까, 특정 단어가 유리할까?
- 반대로 **아무리 돌려도 안 무너지는 프롬프트**가 있나?
- `repetition_penalty` 대신 `no_repeat_ngram_size=3` 을 줘 본다.
  *"같은 3연속 토큰을 두 번 쓰지 마라"* 는 뜻이다. 결과가 어떻게 다른가?
- 벌칙을 **아주 세게(2.5 이상)** 줘 본다. 반복은 사라지는데 대신 무엇이 망가지나?